# Проект 1. Аналитическая записка
**Дисциплина:** Программирование на Python для бизнес-аналитики

**Выполнил:** Губарев Ярослав Игоревич, студент группы БАЦРФ261

## Бизнес-вопрос

**За счет каких факторов выросла выручка от подписок ПРАЙМ в марте 2026 года, какую роль сыграло повышение стоимости тарифов и что происходило с завершениями подписок после изменения цен (оттоком пользователей)?**

В марте выручка заметно выросла относительно февраля в 2026. Я хочу проверить следующий вопрос: что дало основной прирост - количество платежей, разная продолжительность месяцев или новые суммы списаний. Отдельно посмотрю, есть ли признаки ухудшения удержания после повышения цен.

Основное сравнение - **февраль и март 2026 года**. Это соседние полные месяцы до и после изменения сумм платежей. Для понимания динамики на графиках использую январь-июль 2026 года. Август не беру для основного сравнения: данные заканчиваются до конца месяца, к тому же там есть необычный скачок завершений подписок (его причины неизвестны и не исследуются).

**Гипотеза:** основная часть роста выручки в марте 2026 года связана с повышением сумм списаний по подпискам, а не с увеличением количества платежей в день.

## 1. Данные и уровень детализации

Для работы потребуется установленный модуль seaborn (в ТЗ указано, что его можно использовать).

Работаю с таблицами `subscriptions`, `payments` и `tariffs` из учебной базы ПРАЙМ. Данные загружаю через `pandas`, без SQL. В части платежей исходная строка - **один платёж**, в части подписок - **одна подписка**, в справочнике тарифов - **одна версия тарифа на определённый период**.

Для анализа выручки основной уровень детализации - **месяц платежа**. При проверке структуры платежей - **месяц * сумма платежа**. Эти уровни позволяют ответить на вопрос о выручке и не смешивать количество платежей с количеством клиентов. Таблицы платежей и подписок между собой не соединяю: у одной подписки может быть несколько списаний, а приглашённые участники подписки вообще не должны создавать дополнительные строки выручки. Крупную таблицу платежей читаю частями, чтобы не загружать ее целиком в память.

In [ ]:
import sys
from collections import Counter, defaultdict  # понадобятся в задачах
from datetime import date
from decimal import Decimal
import pandas as pd

from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def query(sql: str, **params) -> list[dict]:
    """Выполнить SQL-запрос и вернуть строки списком словарей {колонка: значение}."""
    try:
        with engine.connect() as conn:
            # Моменты времени в базе хранятся с часовым поясом. День события считаем
            # по UTC — так он у всех одинаковый, какие бы настройки ни стояли у вас.
            conn.execute(text("set time zone 'UTC'"))
            result = conn.execute(text(sql), params)
            return [dict(row) for row in result.mappings()]
    finally:
        # Соединение не держим: сервер у всей группы общий, а после сна ноутбука
        # старое соединение всё равно мёртвое.
        engine.dispose()


print("подключились к базе как:", query("select current_user as login")[0]["login"])

In [ ]:
#загружаем таблицу с подписками
subscriptions = pd.read_sql_table("subscriptions", con=engine, schema="prime")
engine.dispose()

#проверяем размер таблицы
print("количество строк:", len(subscriptions))
print("количество столбцов:", len(subscriptions.columns))
display(subscriptions.head())

In [ ]:
#проверяем уникальность подписок
print("количество подписок:", len(subscriptions))
print("уникальных подписок:", subscriptions["subscription_id"].nunique())
print("уникальных владельцев:", subscriptions["owner_client_id"].nunique())
#смотрим пропуски
display(subscriptions.isna().sum())
#проверяем даты оформления
print("первая подписка:", subscriptions["started_at"].min())
print("последняя подписка:", subscriptions["started_at"].max())

## 2. Подписки: проверка данных и динамика

В таблице **333 905** строк, столько же уникальных `subscription_id` и владельцев подписок. Это не количество всех пользователей ПРАЙМ: приглашенные участники могут пользоваться сервисом без собственной подписки. У **167 978** подписок не заполнены дата и причина завершения - их нельзя автоматически считать ошибочными пропусками. Даты оформления охватывают январь 2024 -  август 2026 года.

Сначала посмотрю, сколько подписок оформлялось и завершалось ежемесячно, а затем проверю изменение действующей базы. Изменение базы считаю как новые - завершекнные, последовательно накапливая разницу по месяцам.

In [ ]:
#определяем месяцы оформления и завершения
subscriptions["start_month"] = subscriptions["started_at"].dt.strftime("%Y-%m")
subscriptions["end_month"] = subscriptions["ended_at"].dt.strftime("%Y-%m")
#считаем новые и завершенные подписки
monthly_starts = subscriptions.groupby("start_month").size()
monthly_ends = subscriptions.groupby("end_month").size()
#объединяем результаты в одну таблицу
monthly_subscriptions = pd.DataFrame({"new": monthly_starts, "ended": monthly_ends})
monthly_subscriptions = monthly_subscriptions.fillna(0).astype(int)
display(monthly_subscriptions)

In [ ]:
#считаем разницу между новыми и завершенными подписками
monthly_subscriptions["net_change"] = monthly_subscriptions["new"] - monthly_subscriptions["ended"]
#считаем количество действующих подписок на конец каждого месяца
monthly_subscriptions["active"] = monthly_subscriptions["net_change"].cumsum()
#смотрим последние 12 месяцев
display(monthly_subscriptions.tail(12))

In [ ]:
#находим массовые завершения 22 и 23 августа
mass_ends = subscriptions["ended_at"].isin([pd.Timestamp("2026-08-22", tz="UTC"), pd.Timestamp("2026-08-23", tz="UTC")]).sum()
#оцениваем влияние этих завершений
august_ends = monthly_subscriptions.loc["2026-08", "ended"]
august_new = monthly_subscriptions.loc["2026-08", "new"]
print("массовых завершений:", mass_ends)
print("остальных завершений в августе:", august_ends - mass_ends)
print("условное изменение базы без массовых завершений:", august_new - (august_ends - mass_ends))

### Что показала динамика подписок

До августа подписная база росла. На конец июля 2026 года получилось **192 341** действующая подписка, на конец августа - **167 978**, то есть сокращение на **24 363**. При этом из **34 683** августовских завершений **27 670** пришлись ровно на 00:00 UTC 22 и 23 августа. Без них условное изменение за месяц составило бы **плюс 3307**.

Удалять эти записи нельзя, так как причина массовых завершений неизвестна. Но и сравнивать август с мартом как обычный месяц я не буду. Поэтому основное исследование строю по февральским и мартовским платежам, а дополнительные показатели завершения беру за январь-июль.

## 3. Платежи и выручка

Выручкой называю **сумму платежей со статусом `success` по полю `amount`**. Не включаю неуспешные и возвращенные платежи. Возвраты дополнительно не вычитаю: они уже не попали в выборку `success`, и повторное вычитание уменьшило бы выручку дважды.

`payments_count` - количество успешных платежей, а не уникальных клиентов или подписок. `avg_payment` - выручка, деленная на это количество. Расчет идет по `paid_at` с группировкой по календарному месяцу; соединений, которые могли бы размножить платежи из-за шеринга подписок, здесь нет.

In [ ]:
#загружаем платежи по частям
payments_parts = pd.read_sql_table("payments", con=engine, schema="prime", columns=["paid_at", "status", "amount"], chunksize=100000)
#считаем успешные платежи по месяцам
monthly_parts = []
for part in payments_parts:
    part = part[part["status"] == "success"].copy()
    part["month"] = part["paid_at"].dt.strftime("%Y-%m")
    monthly_parts.append(part.groupby("month")["amount"].agg(["count", "sum"]))

engine.dispose()
#объединяем результаты
monthly_payments = pd.concat(monthly_parts).groupby(level=0).sum()
monthly_payments = monthly_payments.rename(columns={"count": "payments_count", "sum": "revenue"})
display(monthly_payments)

In [ ]:
#считаем средний успешный платежа
monthly_payments["avg_payment"] = monthly_payments["revenue"] / monthly_payments["payments_count"]
#смотрим результаты за 2026 год
display(monthly_payments.loc["2026-01":"2026-08"].round(2))

In [ ]:
#выбираем февраль и март 2026 года
feb = monthly_payments.loc["2026-02"]
mar = monthly_payments.loc["2026-03"]
print("рост выручки, %:", round((mar["revenue"] / feb["revenue"] - 1) * 100, 2)) #считаем изменения показателей
print("рост количества платежей, %:", round((mar["payments_count"] / feb["payments_count"] - 1) * 100, 2))
print("рост среднего платежа, %:", round((mar["avg_payment"] / feb["avg_payment"] - 1) * 100, 2))
#учитываем количество дней в месяцах
print("платежей в день в феврале:", round(feb["payments_count"] / 28, 2))
print("платежей в день в марте:", round(mar["payments_count"] / 31, 2))

### Февраль и март: первоначальное сравнение

В феврале 2026 года было **158 265** успешных платежей на **43 499535 руб.**, в марте - **179 258** на **58 229 042 руб** Выручка выросла на **33,86%**, число платежей - на **13,26%**, а средний платеж - на **18,18%**, примерно с **274,85 до 324,83 руб**

Но в феврале 28 дней, а в марте 31. В расчете на день получается **5 652,32** против **5 782,52** платежа е рост около **2,3%**. Поэтому увеличение числа платежей за месяц частично объясняется его продолжительностью. Теперь проверю, чем вызван скачок среднего платежа.

In [ ]:
#загружаем историю тарифов
tariffs = pd.read_sql_table("tariffs", con=engine, schema="prime")
engine.dispose()
#смотрим стоимость тарифов и даты еее действия
display(tariffs[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]].sort_values(["tariff_id", "valid_from"]))

In [ ]:
#загружаем платежи по частям
payments_parts = pd.read_sql_table("payments", con=engine, schema="prime", columns=["paid_at", "status", "amount"], chunksize=100000)
#считаем количество платежей по суммам
amounts_parts = []
for part in payments_parts:
    part = part[part["status"] == "success"]
    part = part[(part["paid_at"] >= "2026-02-01") & (part["paid_at"] < "2026-04-01")].copy()
    part["month"] = part["paid_at"].dt.strftime("%Y-%m")
    amounts_parts.append(part.groupby(["month", "amount"]).size())
engine.dispose()
payment_amounts = pd.concat(amounts_parts).groupby(level=[0, 1]).sum() #мерджим
display(payment_amounts)

## 4. Тарифы и реальные суммы списаний

Справочник тарифов показывает рост базовой цены с **199 до 249 руб** с марта 2026 года. Для расширенного тарифа есть записи на **399, 429 и 449 руб**, причем сроки действия 429 и 449 руб. пересекаются. Поэтому нельзя без дополнительной проверки выбирать одну цену по дате из справочника.

По фактическим успешным платежам картина однозначнее: в феврале встречаются только **199 руб. (98 241 платёж)** и **399 руб. (60 024)**, в марте - **249 руб (111 289)** и **449 руб (67 969)**. Обе наблюдаемые суммы стали выше на 50 руб., а платежей на 429 руб. в эти месяцы нет. Дальше сравниваю группы платежей по сумме, не пытаясь по одной сумме восстановить `tariff_id` каждого клиента.

## 5. Факторная декомпозиция

Разложу изменение выручки с помощью простого равенства:

**Выручка = количество дней * платежи в день * средний платtж.**
Использую **метод цепных подстановок**: сначала меняю количество дней с 28 на 31 при февральских значениях остальных показателей, затем - число платежей в день, затем - средний платеж. Такой порядок нужен, чтобы эффекты в сумме точно дали фактическое изменение выручки. При другом порядке часть совместного влияния распределилась бы между факторами иначе.

In [ ]:
days_feb = 28
days_mar = 31
#количество платежей в день
freq_feb = feb["payments_count"] / days_feb
freq_mar = mar["payments_count"] / days_mar
#берем среднюю сумму платежа
avg_feb = feb["avg_payment"]
avg_mar = mar["avg_payment"]
#влияние каждого фактора
effect_days = (days_mar - days_feb) * freq_feb * avg_feb
effect_freq = days_mar * (freq_mar - freq_feb) * avg_feb
effect_avg = days_mar * freq_mar * (avg_mar - avg_feb)
#проверяем результаты
revenue_change = mar["revenue"] - feb["revenue"]
print("влияние количества дней:", round(effect_days, 2))
print("влияние платежей в день:", round(effect_freq, 2))
print("влияние среднего платежа:", round(effect_avg, 2))
print("общее изменение выручки:", round(revenue_change, 2))
print("разница при проверке:", round(effect_days + effect_freq + effect_avg - revenue_change, 2))

In [ ]:
#считаем долю платежей по расширенному тарифу
share_feb = payment_amounts.loc[("2026-02", 399.0)] / feb["payments_count"]
share_mar = payment_amounts.loc[("2026-03", 449.0)] / mar["payments_count"]
#влияние повышения цен
effect_price = 50 * mar["payments_count"]
#считаем влияние изменения структуры платежей
effect_structure = 200 * mar["payments_count"] * (share_mar - share_feb)
#проверяем результаты
print("доля расширенного тарифа в феврале, %:", round(share_feb * 100, 2))
print("доля расширенного тарифа в марте, %:", round(share_mar * 100, 2))
print("влияние повышения цен:", round(effect_price, 2))
print("влияние структуры платежей:", round(effect_structure, 2))
print("разница при проверке:", round(effect_price + effect_structure - effect_avg, 2))

In [ ]:
#собираем результаты декомпозиции в таблицу
factors = ["количество дней", "платежи в день", "повышение цен", "структура платежей"]
values = [effect_days, effect_freq, effect_price, effect_structure]
effects = pd.DataFrame({"factor": factors, "effect": values})
#считаем долю каждого фактора в общем изменении выручки
effects["share_percent"] = effects["effect"] / revenue_change * 100
#проводим сумму факторов в качетстве проверки
display(effects.round(2))
print("проверка:", round(effects["effect"].sum() - revenue_change, 2))


### Итог декомпозиции

Сначала разложил выручку на календарные дни, платежи в день и средний платеж. Затем влияние среднего платежа отдельно разделил на **изменение сумм списаний** и **структуру платежей**. Доля более дорогих платежей почти не изменилась: **37,93%** в феврале и **37,92%** в марте.

| Фактор | Вклад, млн руб. | Доля прироста |
|---|---:|---:|
| Количество дней | +4,66 | 31,64% |
| Платежи в день | +1,11 | 7,53% |
| Повышение сумм списаний | +8,96 | 60,85% |
| Структура платежей | −0,003 | −0,02% |
| **Всего** | **+14,73** | **100%** |

Сумма эффектов точно совпала с приростом выручки **14 729 507 руб** Основная часть прироста в нашей модели связана с новыми суммами платежей. Это НЕ оценка того, что произошло бы с поведением клиентов без повышения цены: здесь разложено фактическое изменение выручки.

## 6. Графики

Четыре графика отвечают на разные части вопроса: первый показывает **когда выросла выручка**, второй - **когда изменился средний платёж**, третий - **изменилось ли соотношение групп платежей**, четвёртый - **сколько рублей дал каждый фактор**. Для динамики использую январь-июль 2026 года: август неполный и содержит аномалию завершений подписок.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
#выбираем выручку за январь-июль 2026 года
revenue_plot = monthly_payments.loc["2026-01":"2026-07", "revenue"] / 1000000
#строим КРАСИВЫЙ :))) график
sns.set_theme(style="whitegrid", font="DejaVu Sans")
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(x=revenue_plot.index, y=revenue_plot.values, marker="o", linewidth=2.8, markersize=8, color="#315B7C", ax=ax)
ax.scatter(2, revenue_plot.iloc[2], s=115, color="#16867F", zorder=5)
ax.annotate("43,5 млн ₽", xy=(1, revenue_plot.iloc[1]), xytext=(-18, -25), textcoords="offset points", fontsize=10, color="#315B7C")
ax.annotate("+33,9% к февралю", xy=(2, revenue_plot.iloc[2]), xytext=(2.35, 68), arrowprops={"arrowstyle": "-", "color": "#16867F"}, fontsize=11, color="#16867F", fontweight="bold")
ax.set_title("Выручка от успешных платежей", loc="left", fontsize=15, fontweight="bold", pad=16)
ax.set_xlabel("месяц")
ax.set_ylabel("млн ₽")
ax.set_xticks(range(7), ["янв", "фев", "мар", "апр", "май", "июн", "июл"])
ax.set_ylim(0, 73)
ax.grid(axis="x", visible=False)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
#выбираем средний платеж за январь-июль 2026 года
avg_plot = monthly_payments.loc["2026-01":"2026-07", "avg_payment"]
#строим КРАСИВЫЙ :))) график
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axvspan(1.6, 2.4, color="#E7F3F1", alpha=0.8)
sns.lineplot(x=avg_plot.index, y=avg_plot.values, marker="o", linewidth=2.8, markersize=8, color="#315B7C", ax=ax)
ax.scatter(2, avg_plot.iloc[2], s=115, color="#16867F", zorder=5)
ax.annotate("274,85 ₽", xy=(1, avg_plot.iloc[1]), xytext=(-20, -28), textcoords="offset points", fontsize=10, color="#315B7C")
ax.annotate("324,83 ₽", xy=(2, avg_plot.iloc[2]), xytext=(10, 12), textcoords="offset points", fontsize=11, color="#16867F", fontweight="bold")
ax.set_title("Средняя сумма успешного платежа", loc="left", fontsize=15, fontweight="bold", pad=16)
ax.set_xlabel("месяц")
ax.set_ylabel("руб.")
ax.set_xticks(range(7), ["янв", "фев", "мар", "апр", "май", "июн", "июл"])
ax.set_ylim(255, 345)
ax.grid(axis="x", visible=False)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
#собираем количество платежей по двум группам сумм
payment_mix = pd.DataFrame({"199/249 руб.": [payment_amounts.loc[("2026-02", 199.0)], payment_amounts.loc[("2026-03", 249.0)]], "399/449 руб.": [payment_amounts.loc[("2026-02", 399.0)], payment_amounts.loc[("2026-03", 449.0)]]}, index=["февраль", "март"])
#строим ОЧЕНЬ КРАСИВЫЙ :))) график
payment_mix_plot = payment_mix / 1000
fig, ax = plt.subplots(figsize=(9, 4.5))
payment_mix_plot.plot(kind="bar", stacked=True, color=["#315B7C", "#6AB4AB"], width=0.55, ax=ax, edgecolor="white", linewidth=1)
for i in range(len(payment_mix_plot)):
    basic = payment_mix_plot.iloc[i, 0]
    advanced = payment_mix_plot.iloc[i, 1]
    ax.text(i, basic / 2, f"{basic / (basic + advanced) * 100:.1f}%", ha="center", va="center", color="white", fontsize=12, fontweight="bold")
    ax.text(i, basic + advanced / 2, f"{advanced / (basic + advanced) * 100:.1f}%", ha="center", va="center", color="#163F46", fontsize=12, fontweight="bold")
    ax.text(i, basic + advanced + 5, f"{basic + advanced:.1f} тыс.", ha="center", fontsize=11, fontweight="bold", color="#315B7C")
ax.set_title("Структура успешных платежей: февраль и март", loc="left", fontsize=15, fontweight="bold", pad=16)
ax.set_xlabel("")
ax.set_ylabel("количество платежей, тыс.")
ax.set_ylim(0, 205)
ax.set_xticklabels(["февраль", "март"], rotation=0)
ax.legend(title="сумма платежа", loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
ax.grid(axis="x", visible=False)
sns.despine(ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
#переводим влияние факторов в миллионы рублей
effect_plot = effects["effect"] / 1000000

#строим НУ ОЧЕНЬ КРАСИВЫЙ :))) график
order = ["повышение цен", "количество дней", "платежи в день", "структура платежей"]
colors = {"повышение цен": "#16867F", "количество дней": "#315B7C", "платежи в день": "#81A6BD", "структура платежей": "#C88470"}
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(x=effect_plot, y=effects["factor"], hue=effects["factor"], order=order, palette=colors, legend=False, ax=ax)
for i, factor in enumerate(order):
    amount = effects.loc[effects["factor"] == factor, "effect"].iloc[0] / 1000000
    share = effects.loc[effects["factor"] == factor, "share_percent"].iloc[0]
    if amount >= 0:
        ax.text(amount + 0.15, i, f"+{amount:.2f} млн ₽  ·  {share:.2f}%", va="center", fontsize=10, color="#315B7C", fontweight="bold")
    else:
        ax.text(0.16, i, f"{amount:.3f} млн ₽  ·  {share:.2f}%", va="center", fontsize=10, color="#A8614C")
ax.axvline(0, color="#9FB3C0", linewidth=1.3)
ax.set_title("За счёт чего выросла выручка в марте 2026 года", loc="left", fontsize=15, fontweight="bold", pad=16)
ax.set_xlabel("вклад в прирост выручки, млн ₽")
ax.set_ylabel("")
ax.set_xlim(-0.35, 12.6)
ax.grid(axis="y", visible=False)
sns.despine(ax=ax, left=True)
plt.tight_layout()
plt.show()


### Что видно на графиках

В марте выручка перешла с **43,50 до 58,23 млн руб.** и в следующие месяцы оставалась выше февральского уровня. Средний платёж вырос примерно с **275 до 325 руб.**, при этом соотношение групп платежей почти не поменялось. Последний график показывает, что главный вклад в рост дал именно новый уровень сумм списаний — около **8,96 млн руб.** Влияние структуры почти не видно в масштабе миллионов: это всего **−3,4 тыс. руб.**, точное значение есть в таблице факторов.

## 7. Проверка завершений подписок после повышения цен

Рост выручки сам по себе ещё не говорит, что решение о цене однозначно удачное: более дорогая подписка может сильнее раздражать часть клиентов. Поэтому отдельно смотрю, изменилась ли **доля причины `too_expensive` среди завершённых подписок** после марта.

Это не доля всех клиентов, отказавшихся от сервиса, и не точный показатель оттока. Здесь знаменатель — все завершения за соответствующий месяц.

In [ ]:
#отмены из-за высокой стоимости
price_ends = subscriptions[subscriptions["cancel_reason"] == "too_expensive"].groupby("end_month").size()
#нварь-июль 2026 года
price_check = monthly_subscriptions.loc["2026-01":"2026-07", ["ended"]].copy()
price_check["too_expensive"] = price_ends
#считаем долю отмен из-за высокой стоимости
price_check["share_percent"] = price_check["too_expensive"] / price_check["ended"] * 100
display(price_check.round(2))

In [ ]:
#выбираем данные за январь-июль 2026 года
retention_check = monthly_subscriptions.loc["2026-01":"2026-07"].copy()
#считаем количество подписок на начало месяца
retention_check["active_start"] = retention_check["active"] - retention_check["net_change"]
#считаем отношение завершений к базе на начало месяца
retention_check["ended_percent"] = retention_check["ended"] / retention_check["active_start"] * 100
display(retention_check[["ended", "active_start", "ended_percent"]].round(2))

## 8. Вывод и рекомендации

**Бизнес-вопрос:** почему в марте выросла выручка и что это говорит о тарифной политике ПРАЙМ?

Выручка успешных платежей выросла на **14,73 млн руб. (+33,86%)** относительно февраля. По нашей декомпозиции **60,85%** прироста приходится на повышение фактических сумм списаний, **31,64%** это на дополнительные дни марта и **7,53%** - на изменение платежей в день. Структура платежей практически не повлияла на результат. Получается, что в краткосрочном периоде рост выручки в первую очередь связан с новыми суммами платежей, а не с заметным увеличением платёжной активности. При этом после повышения цен не наблюдается устойчивого роста относительных показателей завершения подписок, хотя говорить об отсутствии влияния на удержание нельзя.

**Предложение бизнесу:** пока не менять суммы по тариафам еще раз только на основании мартовского роста. Сначала продолжать ежемесячно следить за успешными платежами в день, средней суммой, завершениями относительно размера базы и причинами отмен отдельно по тарифам. Если после повышения цен доля завершений начнет устойчиво расти, разбирать это уже на уровне когорт подписчиков, а не только по общим месячным итогам.

**Ограничения:** это факторное разложение фактической выручки, а не эксперимент и не оценка того, что было бы без повышения цен. Мы не оценивали прибыль, возвраты как отдельный денежный поток, жизненный цикл конкретных клиентов и долгосрочный отток. Пересечение дат для расширенного тарифа в справочнике тоже не позволяет без дополнительных проверок однозначно сопоставлять каждую его строку с платежами.